# JSON Data Processing - Restaurant Dataset

This notebook demonstrates working with JSON files containing complex data types (nested objects and arrays).

In [ ]:
# Install and setup Java (for Google Colab)
import os

def install_java():
    !apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
    !java -version

install_java()

In [ ]:
# Install PySpark
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, col, from_unixtime, to_date, year, max as spark_max

spark = SparkSession.builder \
    .appName('Restaurant JSON Analysis') \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

## Load JSON Data

In [ ]:
# Read JSON file
# Update path to your restaurants.json location
restaurants = spark.read.json("/content/sample_data/restaurants.json")

print("Schema of restaurants data:")
restaurants.printSchema()

In [ ]:
# Show sample data
restaurants.show(5, truncate=False)

In [ ]:
# Register as temporary table
restaurants.createOrReplaceTempView("restaurants")

## Basic Queries

### Count of restaurants by cuisine

In [ ]:
# Count distinct cuisines
cuisine_count = spark.sql("""
    SELECT cuisine, COUNT(restaurant_id) as restaurant_count 
    FROM restaurants 
    GROUP BY cuisine
""")

print(f"Total distinct cuisines: {cuisine_count.count()}")

In [ ]:
# Show top cuisines by restaurant count
cuisine_ranking = spark.sql("""
    SELECT cuisine, COUNT(restaurant_id) as restaurant_count 
    FROM restaurants 
    GROUP BY cuisine 
    ORDER BY restaurant_count DESC
""")

cuisine_ranking.show(20)

### Indian restaurants by borough

In [ ]:
# Count Indian restaurants in each borough
indian_by_borough = spark.sql("""
    SELECT borough, COUNT(restaurant_id) as indian_restaurant_count 
    FROM restaurants 
    WHERE cuisine = 'Indian' 
    GROUP BY borough
    ORDER BY indian_restaurant_count DESC
""")

indian_by_borough.show()

### Indian restaurants by street

In [ ]:
# Access nested field (address.street)
indian_by_street = spark.sql("""
    SELECT address.street, COUNT(restaurant_id) as restaurant_count 
    FROM restaurants 
    WHERE cuisine = 'Indian' 
    GROUP BY address.street
    ORDER BY restaurant_count DESC
""")

indian_by_street.show(20)

### Indian restaurants by zipcode

In [ ]:
# Count by zipcode
indian_by_zipcode = spark.sql("""
    SELECT address.zipcode, COUNT(restaurant_id) as cnt 
    FROM restaurants 
    WHERE cuisine = 'Indian' 
    GROUP BY address.zipcode
    ORDER BY cnt DESC
""")

print(f"Total zipcodes with Indian restaurants: {indian_by_zipcode.count()}")
indian_by_zipcode.show()

## Working with Array Objects

### Explode grades array

In [ ]:
# Explode array to create one row per grade
df_array_data = restaurants.select(
    'restaurant_id', 
    'name', 
    explode(restaurants.grades).alias('Grades')
)

print("After exploding grades array:")
df_array_data.show(10, truncate=False)

In [ ]:
# Select specific fields from exploded grades
df2 = df_array_data.select(
    'restaurant_id', 
    'name', 
    'Grades.date', 
    'Grades.grade', 
    'Grades.score'
)

print(f"Total grade records: {df2.count()}")
df2.show(10)

### Find maximum score per restaurant

In [ ]:
# Register as temp table and find max score
df2.createOrReplaceTempView("grade")

max_scores = spark.sql("""
    SELECT restaurant_id, name, MAX(score) as max_score 
    FROM grade 
    GROUP BY restaurant_id, name
    ORDER BY max_score DESC
""")

max_scores.show(20)

### Working with nested date format

In [ ]:
# Access nested date field
df4 = df2.select("restaurant_id", "name", "date.$date", "grade", "score")

df4.show(10)
df4.printSchema()

In [ ]:
# Convert Unix timestamp to date
df5 = df4.select(
    "restaurant_id", 
    "name", 
    from_unixtime(col("$date") / 1000, "yyyy-MM-dd").alias("grade_dt")
)

df5.show(10)

In [ ]:
# Convert to proper date type
df6 = df5.withColumn("grade_dt", to_date(col("grade_dt"), "yyyy-MM-dd"))

df6.printSchema()
df6.show(10)

### Year-wise grade count for specific restaurant

In [ ]:
# Create temp view and query by year
df6.createOrReplaceTempView("dateWiseTable")

year_wise_grades = spark.sql("""
    SELECT 
        name, 
        YEAR(grade_dt) as year, 
        COUNT(*) as grade_count 
    FROM dateWiseTable 
    WHERE name = 'Morris Park Bake Shop' 
    GROUP BY name, YEAR(grade_dt)
    ORDER BY year
""")

year_wise_grades.show()

## Working with Coordinate Arrays

In [ ]:
# Extract longitude and latitude from coord array
df_array_coord = restaurants.select(
    'restaurant_id', 
    'name', 
    restaurants.address.coord[0].alias('longitude'),
    restaurants.address.coord[1].alias('latitude')
)

df_array_coord.show(10)

In [ ]:
# Filter by longitude
df_array_coord.createOrReplaceTempView("coord")

west_restaurants = spark.sql("""
    SELECT * 
    FROM coord 
    WHERE longitude < -74
""")

print(f"Restaurants with longitude < -74: {west_restaurants.count()}")
west_restaurants.show(20)

## Summary Statistics

In [ ]:
# Overall summary
summary = spark.sql("""
    SELECT 
        COUNT(DISTINCT restaurant_id) as total_restaurants,
        COUNT(DISTINCT cuisine) as total_cuisines,
        COUNT(DISTINCT borough) as total_boroughs
    FROM restaurants
""")

summary.show()

In [ ]:
# Borough-wise distribution
borough_distribution = spark.sql("""
    SELECT 
        borough, 
        COUNT(restaurant_id) as restaurant_count,
        COUNT(DISTINCT cuisine) as cuisine_variety
    FROM restaurants 
    GROUP BY borough
    ORDER BY restaurant_count DESC
""")

borough_distribution.show()

In [ ]:
# Stop Spark Session
spark.stop()